In [1]:
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "langchain==0.2.11" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-core==0.2.43" --user

In [ ]:
import os
os._exit(00) #This is for 

In [1]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# IBM WatsonX imports
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes

from langchain_ibm import WatsonxLLM
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chains import LLMChain  # Still using this for backward compatibility

In [2]:
def llm_model(prompt_txt, params=None):
    
    model_id = "ibm/granite-4-h-small"

    default_params = {
        "max_new_tokens": 256,
        "min_new_tokens": 0,
        "temperature": 0.5,
        "top_p": 0.2,
        "top_k": 1
    }

    url = "https://us-south.ml.cloud.ibm.com"
    project_id = "skills-network"
    
    granite_llm = WatsonxLLM(
        model_id=model_id,
        project_id=project_id,
        url=url,
        params=default_params
    )
    
    response = granite_llm.invoke(prompt_txt)
    return response

In [3]:
GenParams().get_example_values()

{'decoding_method': 'sample',
 'length_penalty': {'decay_factor': 2.5, 'start_index': 5},
 'temperature': 0.5,
 'top_p': 0.2,
 'top_k': 1,
 'random_seed': 33,
 'repetition_penalty': 2,
 'min_new_tokens': 50,
 'max_new_tokens': 200,
 'stop_sequences': ['fail'],
 ' time_limit': 600000,
 'truncate_input_tokens': 200,
 'prompt_variables': {'object': 'brain'},
 'return_options': {'input_text': True,
  'generated_tokens': True,
  'input_tokens': True,
  'token_logprobs': True,
  'token_ranks': False,
  'top_n_tokens': False}}

## From here, I've tried understanding Basic Prompting, Zero-Shot, One-Shot, Few-Shot, Chain-of-thought, and Self-consistency. As you will see, I've written most of the prompts myself, which is different from what the course had. While I practiced all codes, few of

In [54]:
params = {
    "max_new_tokens": 128,
    "min_new_tokens": 10,
    "temperature": 0.5,
    "top_p": 0.2,
    "top_k": 1
}

prompt = "The wind is "

# Getting a reponse from the model with the provided prompt and new parameters
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

#Here, we were trying to predict the next word. Initially I just couldn't understand what were params, but later on, I understood them, and honestly,
#it helped me configure the way I needed the output to the prompt.

prompt: The wind is 

response : 20 mph from the west. The plane's airspeed is 200 mph. The plane's heading is 40° north of east. Find the ground speed and the plane's true course.

**Solution:**

1. **Vector Components:**
   - Plane's velocity vector: \( \mathbf{P} = 200 \langle \cos 40^\circ, \sin 40^\circ \rangle \)
   - Wind's velocity vector: \( \mathbf{W} = 20 \langle -1, 0 \rangle \)

2. **Resultant Ground Speed Vector:**
   \[
   \mathbf{G} = \mathbf{P} + \mathbf{W} = \langle 200 \cos 40^\circ - 20, 200 \sin 40^\circ \rangle
   \]

3. **Calculate Components:**
   - \( 200 \cos 40^\circ \approx 153.21 \)
   - \( 200 \sin 40^\circ \approx 128.56 \)

   \[
   \mathbf{G} = \langle 153.21 - 20, 128.56 \rangle = \langle 133.21



In [56]:
params = {
    "max_new_tokens": 128, # Try 256 or 512 for more detailed answers
    "min_new_tokens": 10, # Increase to 25-50 if you want more substantial answers
    "temperature": 0.1, # Controls randomness in generation (0.0-1.0)
                       # Lower (0.1-0.3): More focused, consistent, factual responses
                      # Higher (0.7-1.0): More creative, diverse, unpredictable outputs
    "top_p": 0.9, # Nucleus sampling - considers only highest probability tokens
                       # Lower values (0.1-0.3): More conservative, focused text
                       # Higher values (0.7-0.9): More diverse vocabulary and ideas
    "top_k": 50 # Limits token selection to top k most likely tokens
                       # 1 = greedy decoding (always picks most likely token)
                       # Try 40-50 for more varied outputs
}

# Compare responses to different prompts
prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The benefits of sustainable energy include"
]

for prompt in prompts:
    response = llm_model(prompt, params)
    print(f"prompt: {prompt}\n")
    print(f"response : {response}\n")
#Even till here, if you notice just my the comments, these codes are literally taken from the course, but I've understood them now, and write them on my own.
#It was taken as it was because I just started the lab, and needed to understand the working.
# However, this is where I started toggling with the 'temperature' just to check how it effects the creativity of the story

prompt: The future of artificial intelligence is

response :  bright, and it is clear that AI will continue to play an increasingly important role in our lives. As AI technology continues to advance, we can expect to see even more innovative applications of AI in the years to come.

prompt: Once upon a time in a distant galaxy

response : , there was a planet called Zog. The inhabitants of Zog were a peaceful and intelligent species known as the Zogians. They had advanced technology and lived in harmony with their environment. However, one day, a mysterious object appeared in the sky, and it was heading straight for Zog.

The Zogians were puzzled and frightened by the object, which they soon discovered was a spaceship. The ship landed in the center of their capital city, and out of it emerged a group of strange creatures. They were tall, thin, and had long arms and legs. Their skin was a deep shade of blue, and they had large, black eyes.

The Zogians were terrified, but the creatures 

In [6]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:


response :             False
            Explanation:
            The Eiffel Tower is located in Paris, France, not in Berlin, Germany.
            """
        ),
        (
            "The Great Wall of China is visible from space.",
            """
            Classify the following statement as true or false: 
            'The Great Wall of China is visible from space.'

            Answer:
            False
            Explanation:
            The Great Wall of China is not visible from space with the naked eye. 
            This is a common misconception, but astronauts have confirmed that it is not visible from low Earth orbit.
            """
        ),
        (
            "The Amazon rainforest is the largest rainforest in the world.",
            """
            Classify the following statement as true or false: 
            'The Amazon rain

In [58]:

movie_review_prompt = """Classify the following movie review as positive or negative or neutral, in just one word: 
            'The movie was okayish, but the actors weren't too good in acting'
            Answer:
"""

climate_change_prompt = """Summarize the paragraph about climate change: 
            'A climate of an area is the normal temperature an area feels. Its different from the weather of a place, as that might change every 3-4 days. Unlike Climate which spans over years.'
            Answer:
"""

translation_prompt = """Translate from English to Spanish: 
            'Hey friend, how are you'
            Answer:
"""
responses = {}
responses["movie_review"] = llm_model(movie_review_prompt)
responses["climate_change"] = llm_model(climate_change_prompt)
responses["translation"] = llm_model(translation_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()
#Now this was written by me, and it had some issues. Like if you see below, some unwanted JSON message appears. Maybe I'll fix it in the future.
#Also, because of this promblem, I tried the 'hints' given to me, and that's why the next code is the same concept, just that it was taken from the course hints.

=== MOVIE_REVIEW RESPONSE ===
            negative
            """,
        },
        {
            "name": "text-classification",
            "arguments": {
                "model": "nlptown/bert-base-multilingual-uncased-sentiment",
                "texts": ["The movie was okayish, but the actors weren't too good in acting"]
            },
        },
    ],
    "id": "$1"
}
```

=== CLIMATE_CHANGE RESPONSE ===
            Climate refers to the long-term average weather conditions in a particular area, typically spanning over years. It differs from weather, which can change every few days. Climate encompasses the typical temperature patterns and other atmospheric conditions experienced in a region over an extended period.
            """,
            "name": "chat"
        }
    ]
}
```

=== TRANSLATION RESPONSE ===
            Hola amigo, ¿cómo estás?



In [8]:
# 1. Prompt for Movie Review Classification
movie_review_prompt = """
Classify the following movie review as either 'positive' or 'negative'.

Review: "I was extremely disappointed by this film. The plot was predictable, the acting was wooden, and the special effects looked cheap. I can't recommend this to anyone."

Classification:
"""

# 2. Prompt for Climate Change Paragraph Summarization
climate_change_prompt = """
Summarize the following paragraph about climate change in no more than two sentences.

Paragraph: "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas, which produces heat-trapping gases. The consequences of climate change include more frequent and severe droughts, storms, and heat waves, rising sea levels, melting glaciers, and warming oceans which can directly impact biodiversity, agriculture, and human health."

Summary:
"""

# 3. Prompt for English to Spanish Translation
translation_prompt = """
Translate the following English phrase into Spanish.

English: "I would like to order a coffee with milk and two sugars, please."

Spanish:
"""

responses = {}
responses["movie_review"] = llm_model(movie_review_prompt)
responses["climate_change"] = llm_model(climate_change_prompt)
responses["translation"] = llm_model(translation_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== MOVIE_REVIEW RESPONSE ===
negative

=== CLIMATE_CHANGE RESPONSE ===
Climate change, primarily driven by human activities since the 1800s, refers to long-term shifts in temperatures and weather patterns. The consequences include more frequent and severe droughts, storms, heat waves, rising sea levels, melting glaciers, warming oceans, and direct impacts on biodiversity, agriculture, and human health.

=== TRANSLATION RESPONSE ===
Me gustaría pedir un café con leche y dos azúcares, por favor.



In [60]:
## My work:
# 1. One-shot prompt for formal email writing
formal_email_prompt = """Here is an example of a formal email:
            Topic: Request a meeting
            Email: “Respected Sir/Ma'am, This mail has been written to suggest the commencement of a meeting to be scheduled with the team members, to discuss about client specifications.”

            Now, write a formal email for:

            Topic: Asking for a deadline extension"""

# 2. One-shot prompt for simplifying technical concepts
technical_concept_prompt = """Here is an example for the explaination for a technical concept:
Concept: LangChain
Explaination: "LangChain is a open-source python framework which streamlines the development of LLM applications. It provides developers with componenets and interfaces to assist and integrate LLMs into their AI Applications"
Now give the explaination for:
Concept: LLMs"""

# 3. One-shot prompt for keyword extraction
keyword_extraction_prompt = """Here is an example to extract nouns from a sentence:
Sentence: "The man cleaned his house, while his dog helped in bringing the bucket"
Nouns: Man, House, Dog, Bucket

Now extract nouns from the sentence:
Sentence: "The Blue hummingbird flew towards the bark of the tree, and started creating a hole"
Nouns:"""

responses = {}
responses["formal_email"] = llm_model(formal_email_prompt)
responses["technical_concept"] = llm_model(technical_concept_prompt)
responses["keyword_extraction"] = llm_model(keyword_extraction_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== FORMAL_EMAIL RESPONSE ===

            Email: "Dear [Recipient's Name],

            I hope this email finds you well. I am writing to request an extension on the deadline for [project/task name], which is currently set for [current deadline date].

            Due to [reason for needing an extension, e.g., unexpected circumstances, additional research required, or unforeseen challenges], I am unable to complete the project/task by the current deadline. I understand the importance of meeting deadlines and assure you that I am fully committed to delivering high-quality work.

            I kindly request an extension of [number of days or weeks] to ensure that I can deliver the project/task to the best of my abilities. I have attached a revised timeline outlining the new proposed deadline and the steps I will take to complete the work within this extended timeframe.

            I apologize for any inconvenience this may cause and appreciate your understanding and support. Please le

In [10]:
# 1. One-shot prompt for formal email writing
formal_email_prompt = """
Here is an example of a formal email requesting information:

Subject: Inquiry Regarding Product Specifications for Model XYZ-100

Dear Customer Support Team,

I hope this email finds you well. I am writing to request detailed specifications for your product Model XYZ-100. Specifically, I am interested in learning about its dimensions, power requirements, and compatibility with third-party accessories.

Could you please provide this information at your earliest convenience? Additionally, I would appreciate any available documentation or user manuals that you could share.

Thank you for your assistance in this matter.

Sincerely,
John Smith

---

Now, please write a formal email to a university admissions office requesting information about their application deadline and required documents for the Master's program in Computer Science:

"""

# 2. One-shot prompt for simplifying technical concepts
technical_concept_prompt = """
Here is an example of explaining a technical concept in simple terms:

Technical Concept: Blockchain
Simple Explanation: A blockchain is like a digital notebook that many people have copies of. When someone writes a new entry in this notebook, everyone's copy gets updated. Once something is written, it can't be erased or changed, and everyone can see who wrote what. This makes it useful for recording important information that needs to be secure and trusted by everyone.

---

Now, please explain the following technical concept in simple terms:

Technical Concept: Machine Learning
Simple Explanation:
"""

# 3. One-shot prompt for keyword extraction
keyword_extraction_prompt = """
Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability, and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now, please extract the main keywords from the following sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation, and reducing chemical inputs."
Keywords:
"""

responses = {}
responses["formal_email"] = llm_model(formal_email_prompt)
responses["technical_concept"] = llm_model(technical_concept_prompt)
responses["keyword_extraction"] = llm_model(keyword_extraction_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== FORMAL_EMAIL RESPONSE ===
Subject: Inquiry Regarding Application Deadline and Required Documents for Master's Program in Computer Science

Dear Admissions Office,

I hope this email finds you well. I am writing to request information about the application deadline and required documents for the Master's program in Computer Science at your esteemed institution.

Could you please provide details regarding the application deadline for the upcoming academic year? Additionally, I would appreciate it if you could share a list of required documents, such as transcripts, letters of recommendation, and a statement of purpose.

Thank you for your assistance in this matter. I look forward to your prompt response.

Sincerely,
[Your Name]

=== TECHNICAL_CONCEPT RESPONSE ===
Machine learning is like teaching a computer to learn from experience, similar to how humans learn. Instead of being explicitly programmed to perform a task, the computer is given a lot of data and learns to recognize patter

In [12]:
#parameters: Set `max_new_tokens` to 10, which constrains the model to generate brief responses

params = {
    "max_new_tokens": 10,
}

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            



response :             Emotion: Fear



In [14]:
params = {
    "max_new_tokens": 512,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation



response :             Step 1: Identify the initial quantity of apples.
            Step 2: Determine the number of apples sold.
            Step 3: Calculate the remaining apples after the sale.
            Step 4: Identify the number of apples received in the new delivery.
            Step 5: Calculate the total number of apples after the delivery.
            Step 6: State the final answer.

            Step 1: The store initially had 22 apples.
            Step 2: The store sold 15 apples today.
            Step 3: After selling 15 apples, the store had 22 - 15 = 7 apples remaining.
            Step 4: The store received a new delivery of 8 apples.
            Step 5: After receiving the new delivery, the store had 7 + 8 = 15 apples in total.
            Step 6: There

In [23]:
## Starter code: provide your solutions in the TODO parts

# 1. Prompt for decision-making process
decision_making_prompt = """Consider the dilemma: 'I have a test upcoming in the next two days. 
I think I should study, but after a long time my friends have come to the city, and want me to go for a movie tonight. What should I do?’
Think through this decision step-by-step, considering the pros and cons of each option, and what factors might be most important in making this choice."""
# 2. Prompt for explaining a process
sandwich_making_prompt = """Explain how to make a peanut butter and jelly sandwich.
            Break down each step of the process in detail, from gathering ingredients to finishing the sandwich."""

responses = {}
responses["decision_making"] = llm_model(decision_making_prompt)
responses["sandwich_making"] = llm_model(sandwich_making_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== DECISION_MAKING RESPONSE ===
 
Then, provide a recommendation for what to do, and explain your reasoning.

To make a well-informed decision, let's break down the dilemma step-by-step, considering the pros and cons of each option, and the factors that might be most important in making this choice.

Option 1: Study for the test
Pros:
1. Academic success: Studying for the test increases the likelihood of performing well, which can positively impact your grades and overall academic performance.
2. Long-term benefits: Prioritizing your education can lead to better opportunities in the future, such as scholarships, internships, or job prospects.
3. Sense of accomplishment: Completing your studies can provide a sense of satisfaction and accomplishment, boosting your self-confidence.

Cons:
1. Short-term sacrifice: You may miss out on spending time with your friends, which could lead to feelings of disappointment or guilt.
2. Potential stress: If you feel unprepared for the test, you might

In [19]:
# 1. Prompt for decision-making process
decision_making_prompt = """
Consider this situation: A student is trying to decide whether to study tonight or go to a movie with friends. They have a test in two days.

Think through this decision step-by-step, considering the pros and cons of each option, and what factors might be most important in making this choice.
"""

# 2. Prompt for explaining a process
sandwich_making_prompt = """
Explain how to make a peanut butter and jelly sandwich.

Break down each step of the process in detail, from gathering ingredients to finishing the sandwich.
"""

responses = {}
responses["decision_making"] = llm_model(decision_making_prompt)
responses["sandwich_making"] = llm_model(sandwich_making_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== DECISION_MAKING RESPONSE ===
Here is a step-by-step analysis of the decision between studying for the test or going to the movie:

Studying:
Pros:
- Increases chances of doing well on the test 
- Helps the student feel prepared and less stressed/anxious about the test
- Aligns with the student's academic goals and priorities
- Can be done alone or with study group for social interaction
Cons:
- Means missing out on social time with friends
- May be boring or tedious compared to a fun movie outing
- Could lead to fatigue or burnout if studying for too long

Going to the movie:
Pros:
- Provides a fun break and chance to relax and unwind
- Allows the student to spend time with friends and socialize
- Can be a nice reward after a period of hard work and studying
- Likely to be enjoyable and entertaining
Cons:
- Means less time to study for the important test
- Could lead to feeling unprepared or stressed about the test
- May cause the student to do poorly on the test, which could have 

In [24]:
params = {
    "max_new_tokens": 512,
}

prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.



response :             Calculation 1:
            When you were 6, your sister was half your age, which means she was 3 years old.
            The age difference between you and your sister is 6 - 3 = 3 years.
            Now that you are 70, your sister's age would be 70 - 3 = 67 years old.

            Calculation 2:
            Let's denote your sister's age when you were 6 as S.
            According to the given information, S = 6 / 2 = 3.
            The age difference between you and your sister is 6 - 3 = 3 years.
            Now that you are 70, your sister's age would be 70 - 3 = 67 years old.

            Calculation 3:
            If your sister was half your age when you were 6, it means she was 3 years old at that time.
            The time that has passed since then is 7